(centile-positions)=
# Centile Position Calculation

### gamlss_main.Gamlss.<br>**get_centile_position**

```python
get_centile_position(test_data, train_data, observed_values=None, func_name=None, mode=None,
                    use_transfer=False, gam_refit_path=None, params_to_adjust=None,
                    adjustment_covariates=None)
```

&nbsp;&nbsp;&nbsp;&nbsp;Calculates what percentile each observed value corresponds to within the model's estimated distribution. This is the inverse of centile calculation.

&nbsp;&nbsp;&nbsp;&nbsp;<span style="background-color: #f5f5f5; padding: 4px 8px; display: inline-block; width: 90%;">**Parameters:**</span>

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**test_data** : ***pandas.DataFrame*** <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;DataFrame containing the test data with predictor variables

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**train_data** : ***pandas.DataFrame*** <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;DataFrame containing the training data

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**observed_values** : ***array-like***, *optional* <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Specific values to calculate centile positions for (if None, uses test_data y values)

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**func_name** : ***str***, *optional* <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Not needed; leave it out (the model's own family is always used). Still accepted so that older code keeps working (a different family raises an error); passing it shows a FutureWarning and it will be removed in the next version

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**mode** : ***str***, *optional* <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Calculation method: 'individual', 'matrix', 'average', or None for auto-selection

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**use_transfer** : ***bool***, *default=False* <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;If True, use site transfer adjustments for percentile calculation.

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**gam_refit_path** : ***str***, *optional* <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Path to saved adjustment model (.rds file). Required if `use_transfer=True`.

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**params_to_adjust** : ***list of str***, *optional* <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Parameters the transfer adjusted (`site_transfer`'s `params_to_adjust`). Only used if `use_transfer=True`. <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;If None (default), read from the saved adjustment model itself; if given, it must match what the transfer adjusted (an error is raised otherwise).

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**adjustment_covariates** : ***list of str***, *optional* <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Covariate columns the transfer was fitted with (`site_transfer`'s `adjustment_covariates`). Only used if `use_transfer=True`. <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Required when the adjustment model has covariate terms; leaving it out then raises an error (default is None).

&nbsp;&nbsp;&nbsp;&nbsp;<span style="background-color: #f5f5f5; padding: 4px 8px; display: inline-block; width: 90%;">**Returns:**</span>

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**percentiles** : ***numpy.ndarray*** <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Percentile positions (0-100 scale) with shape depending on mode

&nbsp;&nbsp;&nbsp;&nbsp;<span style="background-color: #f5f5f5; padding: 4px 8px; display: inline-block; width: 90%;">**Mode Options:**</span>

&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**'individual'** : One percentile per test data row using its own parameters <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**'matrix'** : Each observed value against each parameter set (2D output) <br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;**'average'** : Single percentile per observed value using average parameters

<br>


**Examples:**

In [39]:
import numpy as np

# Get percentile for all test data points
percentiles = gamlss_model.get_centile_position(df_test, df_train)
print(f"Shape of percentiles: {percentiles.shape}")

# Get percentile for specific values (mode='matrix' is automatic for specific values)
specific_values = np.array([10.5, 15.2, 20.1])
specific_percentiles = gamlss_model.get_centile_position(df_test, df_train, observed_values=specific_values)
print(f"Shape of specific percentiles: {specific_percentiles.shape}")

# Use mode='average' for a single percentile per value, based on the average of all parameters
avg_percentiles = gamlss_model.get_centile_position(df_test, df_train, observed_values=specific_values, mode='average')
print(f"Shape of average percentiles: {avg_percentiles.shape}")

Shape of percentiles: (470,)
Shape of specific percentiles: (470, 3)
Shape of average percentiles: (3,)


In [40]:
### A sanity test on above functions working
# Calculate specific centiles (e.g., 10th, 50th, 90th)
test_centiles = [10, 50, 90]
centile_values = gamlss_model.centiles(df_test, df_train, 
                                      centiles=test_centiles)

# Select a single test sample for simplicity
sample_idx = 0  # First sample
sample_centile_values = centile_values[sample_idx, :]

# Use get_centile_position on those values
calculated_positions = gamlss_model.get_centile_position(
    df_test.iloc[[sample_idx]], df_train,
    observed_values=sample_centile_values,
    mode='matrix'  # Since we're giving multiple values
)

# Print results
print("Values at centiles:", sample_centile_values)
print("Expected centile positions:", test_centiles)
print("Calculated centile positions:", calculated_positions[0])

# Calculate difference
print("Difference:", np.abs(calculated_positions[0] - test_centiles))

Values at centiles: [22.70404629 24.16216192 25.63883369]
Expected centile positions: [10, 50, 90]
Calculated centile positions: [10.00005061 50.         89.99999664]
Difference: [5.06078255e-05 0.00000000e+00 3.35895342e-06]
